In [ ]:
# ==========================================
# NLGCL Kaggle 运行指南 (All-in-One)
# ==========================================
# 本 Notebook 用于在 Kaggle 环境中配置并运行 NLGCL 模型。
# 
# ## 前置条件
# 1. **GPU 环境**: 请确保 Notebook 设置中 Accelerator 选择 GPU (如 GPU P100 或 T4)。
# 2. **数据集**: 请上传本地 `dataset` 文件夹到 Kaggle Datasets (命名推荐 `nlgcl-dataset`)，并挂载到 Notebook 中。
#    - Kaggle 挂载路径通常为 `/kaggle/input/nlgcl-dataset`。
#
# ------------------------------------------
# STEP 1: 环境检测与代码准备
# ------------------------------------------
import os

# 定义你的 GitHub 仓库地址 (请替换为你的实际地址)
GITHUB_REPO_URL = "https://github.com/Jinfeng-Xu/NLGCL.git" 
REPO_NAME = "NLGCL" # 仓库名

if os.path.exists('/kaggle/working'):
    # 在 Kaggle 环境下
    print("Detected Kaggle environment.")
    
    # 克隆代码 (如果不存在)
    if not os.path.exists(REPO_NAME):
        cmd = f"git clone {GITHUB_REPO_URL} {REPO_NAME}"
        print(f"Executing: {cmd}")
        os.system(cmd)
    
    # 进入代码目录
    if os.path.exists(REPO_NAME):
        os.chdir(REPO_NAME)
        print(f"Changed directory to {os.getcwd()}")
else:
    print("Not in Kaggle environment, assuming local run.")
    # 如果在本地，不需要 clone，直接运行即可

# ------------------------------------------
# STEP 2: 安装依赖
# ------------------------------------------
print("Installing dependencies...")

# [优化] 预先解决 colorama 版本冲突，避免 pip 长时间回溯
!pip install "colorama>=0.4.6"

# [优化] 设置并行编译的核心数，加速源码编译过程
os.environ['MAX_JOBS'] = '4'

# 强制升级/安装 recbole
!pip install --upgrade recbole

# 安装其他依赖
!pip install -r requirements.txt

# 安装 PyG 及其相关依赖
# Kaggle 的 PyTorch 版本可能很新 (如 2.x)，会导致官方 wheel 找不到。
# 我们尝试查找 wheel，如果失败则回退到源码编译安装。
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

try:
    if torch.cuda.is_available():
        cuda_version = torch.version.cuda
        print(f"CUDA Version: {cuda_version}")
        
        # 尝试构建 wheel 链接
        torch_version = torch.__version__.split('+')[0]
        cuda_version_clean = cuda_version.replace('.', '')
        wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version_clean}.html"
        
        print(f"Attempting to install PyG binaries from: {wheel_url}")
        
        # 移除 pyg_lib (通常会导致安装失败且非必须)，只安装核心扩展
        packages = ["torch_scatter", "torch_sparse", "torch_cluster", "torch_spline_conv"]
        command = f"pip install {' '.join(packages)} -f {wheel_url}"
        
        exit_code = os.system(command)
        
        if exit_code != 0:
            print("Binary installation failed. Falling back to source compilation (This may take a few minutes)...")
            # 回退：直接安装，让 pip 自己解决（使用 MAX_JOBS 加速编译）
            !pip install torch_scatter torch_sparse torch_cluster torch_spline_conv
    else:
        print("CUDA not available, installing CPU PyG dependencies...")
        !pip install torch_scatter torch_sparse torch_cluster torch_spline_conv

    # 最后安装 torch_geometric
    !pip install torch_geometric

except Exception as e:
    print(f"An error occurred during PyG installation: {e}")
    print("Trying fallback installation...")
    !pip install torch_scatter torch_sparse torch_cluster torch_spline_conv torch_geometric

# ------------------------------------------
# STEP 3: 数据集准备 (自动寻找与链接)
# ------------------------------------------
import shutil

# 自动寻找 Kaggle Input 中的 dataset 目录
def find_dataset_root(search_path='/kaggle/input'):
    for root, dirs, files in os.walk(search_path):
        if 'yelp.inter' in files:
            # 找到了 yelp.inter，说明它的父目录是 yelp
            # 如果父目录叫 yelp，则 dataset root 是 root 的上一级
            if os.path.basename(root) == 'yelp':
                return os.path.dirname(root)
            # 否则可能结构已经扁平化，我们直接返回上一级模拟一下
            return os.path.dirname(root)
    return None

if os.path.exists('/kaggle/working'):
    current_dataset_link = 'dataset'
    
    # 1. 寻找挂载的数据集
    found_path = find_dataset_root()
    if not found_path:
        # 尝试默认路径
        potential_paths = [
            "/kaggle/input/nlgcl-dataset",
            "/kaggle/input/nlgcl-dataset/dataset",
            "/kaggle/input/dataset"
        ]
        for p in potential_paths:
            if os.path.exists(os.path.join(p, 'yelp', 'yelp.inter')):
                found_path = p
                break
    
    if found_path:
        print(f"Found dataset at: {found_path}")
        
        # 2. 清理旧的 dataset 目录/链接
        if os.path.exists(current_dataset_link):
            if os.path.islink(current_dataset_link):
                os.unlink(current_dataset_link)
            else:
                shutil.rmtree(current_dataset_link)
        
        # 3. 复制数据集
        print("Copying dataset to working directory (required for write access to generate processed files)...")
        shutil.copytree(found_path, current_dataset_link)
        print("Dataset copied successfully.")
        
    else:
        print("Error: Could not find 'yelp/yelp.inter' in /kaggle/input. Please verify your dataset upload.")
        # 列出 input 目录帮助调试
        !find /kaggle/input -maxdepth 3
else:
    print("Local environment, skipping dataset setup.")

!ls -l dataset/yelp/

# ------------------------------------------
# STEP 4: 强制写入过滤配置 (覆盖 yelp.yaml)
# ------------------------------------------
# 为什么之前的运行没有过滤？
# 1. Kaggle 环境可能未同步最新 Git 代码。
# 2. 必须覆盖配置文件确保过滤生效。

yelp_config_content = """
load_col:
  inter: [user_id, item_id, rating]
ITEM_ID_FIELD: item_id
RATING_FIELD: rating

# 核心过滤配置：过滤交互少于 15 次的用户和物品
user_inter_num_interval: "[15,inf)"
item_inter_num_interval: "[15,inf)"

val_interval:
  rating: "[3,inf)"

warm_up_step: 40
"""

os.makedirs('properties', exist_ok=True)
with open('properties/yelp.yaml', 'w') as f:
    f.write(yelp_config_content)
print("Updated properties/yelp.yaml with filtering rules.")

# ------------------------------------------
# STEP 5: 运行训练
# ------------------------------------------
# 清理旧的缓存，强制重新处理数据
if os.path.exists("saved"):
    import glob
    for pth in glob.glob("saved/*.pth"):
        os.remove(pth)

print("Starting training with correct configuration...")
# 检查 recbole 是否安装成功
try:
    import recbole
    print(f"RecBole version: {recbole.__version__}")
except ImportError:
    print("RecBole still not installed? Trying again...")
    !pip install recbole

!python main.py --dataset=yelp